# 05 - Data Quality Metrics (Gold)

One row per checked table for this run.
`quality_status` is FAIL when any checked count is greater than zero.

In [0]:
import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType
)

from configs.project_config import (
    SILVER_TABLES, GOLD_TABLES, BUSINESS_KEY_COLS, VALID_QUARTERS, CROP_GROUPS
)

run_id = str(uuid.uuid4())
checked = [
    SILVER_TABLES['crop_production_enriched'],
    GOLD_TABLES['province_quarter_summary'],
    GOLD_TABLES['crop_group_trends'],
    GOLD_TABLES['crop_province_performance'],
    GOLD_TABLES['ml_crop_forecast_features']
]

rows = []
for table_name in checked:
    df = spark.table(table_name)
    row_count = df.count()

    if set(BUSINESS_KEY_COLS).issubset(df.columns):
        duplicate_count = row_count - df.select(*BUSINESS_KEY_COLS).distinct().count()
    elif set(['province', 'year', 'quarter']).issubset(df.columns) and 'crop' not in df.columns:
        duplicate_count = row_count - df.select('province', 'year', 'quarter').distinct().count()
    elif set(['year', 'quarter', 'crop_group']).issubset(df.columns):
        duplicate_count = row_count - df.select('year', 'quarter', 'crop_group').distinct().count()
    else:
        duplicate_count = 0

    null_count = 0
    for col_name in [c for c in ['province', 'crop', 'year', 'quarter', 'production'] if c in df.columns]:
        null_count += df.filter(F.col(col_name).isNull()).count()

    invalid_province_count = (
        df.filter(F.col('province').isNull()).count() if 'province' in df.columns else 0
    )
    invalid_crop_count = (
        df.filter(F.col('crop').isNull() | (F.trim(F.col('crop')) == '')).count()
        if 'crop' in df.columns else 0
    )
    invalid_quarter_count = (
        df.filter(~F.col('quarter').isin(VALID_QUARTERS)).count()
        if 'quarter' in df.columns else 0
    )
    if 'crop_group' in df.columns:
        invalid_crop_count += df.filter(~F.col('crop_group').isin(CROP_GROUPS)).count()

    failed = any(v > 0 for v in [
        duplicate_count, null_count, invalid_province_count,
        invalid_crop_count, invalid_quarter_count
    ])
    rows.append((
        run_id,
        table_name,
        row_count,
        int(duplicate_count),
        int(null_count),
        int(invalid_province_count),
        int(invalid_crop_count),
        int(invalid_quarter_count),
        'FAIL' if failed else 'PASS'
    ))

schema = StructType([
    StructField('run_id', StringType(), False),
    StructField('table_name', StringType(), False),
    StructField('row_count', LongType(), False),
    StructField('duplicate_count', LongType(), False),
    StructField('null_count', LongType(), False),
    StructField('invalid_province_count', LongType(), False),
    StructField('invalid_crop_count', LongType(), False),
    StructField('invalid_quarter_count', LongType(), False),
    StructField('quality_status', StringType(), False)
])

metrics = (
    spark.createDataFrame(rows, schema)
    .withColumn('run_timestamp', F.current_timestamp())
)

target = GOLD_TABLES['data_quality_metrics']
(
    metrics.write
    .format('delta')
    .mode('append')
    .option('mergeSchema', 'true')
    .saveAsTable(target)
)

print(f'Appended run {run_id} to {target}')
metrics.show(truncate=False)